In [1]:
import ee
import geemap

In [ ]:

ee.Initialize(project='YOUR_GEE_PROJECT')  # Optionnel si tu as un projet GCP, sinon ee.Initialize() suffit souvent

In [ ]:

Map = geemap.Map(basemap='Esri.WorldImagery')

In [ ]:

globcover = ee.Image('ESA/GLOBCOVER_L4_200901_200912_V2_3')

In [5]:
# Sélectionner la bande principale (landcover)
landcover = globcover.select('landcover')

In [ ]:

vis_params = {
    'min': 0,
    'max': 220,
    'palette': [
        '000000', 'aaaaaa', 'ffffff', '6a6a6a', '00ff00', 'b4ff92', 'e6004d',
        '9c9c9c', 'c4aa74', 'ffff99', 'c6c64d', 'a87000', 'ff0000', '00aaff',
        '0033ff', 'e6e600', 'a6a600', 'e64c00', 'ff99ff', 'b4b4ff', '00ffff',
        'aaf0f8'
    ]  #
}

In [ ]:
portugal = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017').filter(ee.Filter.eq('country_na', 'Portugal'))


clipped_portugal = landcover.clip(portugal)

In [ ]:

Map.addLayer(clipped_portugal, vis_params, 'ESA GlobCover 2009 - 300m')
Map.centerObject(portugal, 7)  

Map

Map(center=[39.661027428553666, -7.972070540756807], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
'''# Export vers Google Drive
task = ee.batch.Export.image.toDrive(
    image=clipped_portugal,
    description='andCover_300m_Portugal_continental',
    folder='GEE_Exports',              
    fileNamePrefix='landcover_portugal_300m',
    region=portugal.geometry().bounds(),
    scale=300,                          # Résolution native
    maxPixels=1e13,                     # Augmente si erreur "too many pixels"
    crs='EPSG:4326'                     # WGS84
)

task.start()
print("Export lancé ! ")
print("Statut : ", task.status())     # Pour suivre
'''

# check if we keept all the fire in continental Portugal

In [13]:
import pandas as pd
Fire = pd.read_csv("Data/Fire_Data/filtered_fire_data.csv")

In [ ]:

features = []
for index, row in Fire.iterrows():

    point = ee.Geometry.Point([row['longitude'], row['latitude']])
    feature = ee.Feature(point, {'fire_id': index}) 
    features.append(feature)


fire_points = ee.FeatureCollection(features)

Map.addLayer(fire_points, {'color': 'pink'}, 'Points Incendies (Fire)')
Map

Map(bottom=669.0, center=[44.84029065139799, -308.32031250000006], controls=(WidgetControl(options=['position'…